In [1]:
import csv
import requests
import time

DEEPL_API_KEY = "5b27a3ee-bf63-46f6-b0da-1450273772d4"  # ← ここにAPIキーを入れる
DEEPL_ENDPOINT = "https://api.deepl.com/v2/translate"  # 有料版は api.deepl.com に変更
SOURCE_CSV = "C:\\Users\\nhaya\\OneDrive\\Desktop\\JPHACKS2025\\hs_2503\\csv\\shelter_hiroshima_chinese.csv"
OUTPUT_CSV = "translated.csv"

# DeepL翻訳関数
def deepl_translate(texts, target_lang="EN", source_lang=None, retries=5):
    results = []
    for text in texts:
        for attempt in range(retries):
            try:
                data = {
                    "auth_key": DEEPL_API_KEY,
                    "text": text,
                    "target_lang": target_lang,
                }
                if source_lang:
                    data["source_lang"] = source_lang
                r = requests.post(DEEPL_ENDPOINT, data=data, timeout=60)
                if r.status_code == 200:
                    results.append(r.json()["translations"][0]["text"])
                    break
                elif r.status_code in (429, 503):
                    # レートリミット時はリトライ
                    time.sleep(min(2**attempt, 30))
                else:
                    raise Exception(f"Error: {r.text}")
            except Exception as e:
                if attempt == retries - 1:
                    print("Failed:", text, e)
                    results.append(text)
    return results


# CSV読み込み
with open(SOURCE_CSV, encoding="utf-8") as f:
    reader = list(csv.reader(f))

header = reader[0]  # 1行目をヘッダーとして残す
data_rows = reader[1:]

# 各セルを翻訳（ヘッダー以外）
for i, row in enumerate(data_rows):
    for j, cell in enumerate(row):
        if cell.strip():
            translated = deepl_translate([cell], target_lang="EN", source_lang="ZH")
            row[j] = translated[0]

# 新しいCSVに保存
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)   # ヘッダーは翻訳せずそのまま
    writer.writerows(data_rows)

print(f"✅ Translated CSV saved to: {OUTPUT_CSV}")


✅ Translated CSV saved to: translated.csv
